In [1]:
import pandas as pd

In [2]:
orders = pd.read_csv("../data/cleaned/orders_clean.csv")
order_items = pd.read_csv("../data/cleaned/order_items_clean.csv")
customers = pd.read_csv("../data/cleaned/customers_clean.csv")
sellers = pd.read_csv("../data/cleaned/sellers_clean.csv")

In [3]:
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Customers:", customers.shape)
print("Sellers:", sellers.shape)

Orders: (99441, 8)
Order Items: (112650, 7)
Customers: (99441, 5)
Sellers: (3095, 4)


In [4]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)

orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [5]:
orders[[
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "delivery_days"
]].head()

,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,13.0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,9.0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-02 00:28:42,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-16 18:17:02,2.0


In [6]:
orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
)

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

In [7]:
orders[[
    "order_id",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_delay_days"
]].head()

,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,-8.0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,-18.0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,-13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,-10.0


In [8]:
print("Orders columns:")
print(orders.columns.tolist())

print("\nOrder Items columns:")
print(order_items.columns.tolist())

print("\nCustomers columns:")
print(customers.columns.tolist())

print("\nSellers columns:")
print(sellers.columns.tolist())

Orders columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'delivery_delay_days']

Order Items columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Customers columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Sellers columns:
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


In [9]:
order_items[["order_id", "price", "freight_value"]].head()

,order_id,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14


In [10]:
order_items["item_total_value"] = (
    order_items["price"] + order_items["freight_value"]
)

In [11]:
order_items[
    ["order_id", "price", "freight_value", "item_total_value"]
].head()

,order_id,price,freight_value,item_total_value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04


In [12]:
order_value = (
    order_items
    .groupby("order_id")["item_total_value"]
    .sum()
    .reset_index()
)

In [13]:
order_value.head()

,order_id,item_total_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


In [14]:
orders = orders.merge(
    order_value,
    on="order_id",
    how="left"
)

In [15]:
orders = orders.rename(
    columns={"item_total_value": "total_order_value"}
)

In [16]:
orders[
    ["order_id", "total_order_value"]
].head(10)

,order_id,total_order_value
0,e481f51cbdc54678b7cc49136f2d6af7,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,28.62
5,a4591c265e18cb1dcee52889e2d8acc3,175.26
6,136cce7faa42fdb2cefd53fdc79a6098,65.95
7,6514b8ad8028c9f2cc2374ded245783f,75.16
8,76c6e866289321a7c93b82b54852dc33,35.95
9,e69bfb5eb88e0ed6a785585b27e16dbf,169.76


In [17]:
orders["total_order_value"].isna().sum()

np.int64(775)

In [18]:
missing_order_value = orders[
    orders["total_order_value"].isna()
]

missing_order_value[
    ["order_id", "order_status"]
].head(20)

,order_id,order_status
266,8e24261a7e58791d10cb1bf9da94df5c,unavailable
586,c272bcd21c287498b4883c7512019702,unavailable
687,37553832a3a89c9b2db59701c357ca67,unavailable
737,d57e15fb07fd180f06ab3926b39edcd2,unavailable
1130,00b1cb0320190ca0daa2c88b35206009,canceled
1160,2f634e2cebf8c0283e7ef0989f77d217,unavailable
1579,ee0db22a8e742b752914016708470ec8,unavailable
1801,ed3efbd3a87bea76c2812c66a0b32219,canceled
1826,6ad57aecbae806a7e9cc2cdb6b380711,unavailable
1868,df8282afe61008dc26c6c31011474d02,canceled


In [19]:
missing_order_value["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [20]:
missing_ids = orders.loc[
    orders["total_order_value"].isna(),
    "order_id"
]

order_items[
    order_items["order_id"].isin(missing_ids)
].shape

(0, 8)

In [21]:
order_items[
    order_items["order_id"].isin(missing_ids)
]["order_id"].nunique()

0

In [22]:
print("Total orders:", len(orders))
print("Missing total_order_value:", orders["total_order_value"].isna().sum())
print("Available total_order_value:", orders["total_order_value"].notna().sum())

Total orders: 99441
Missing total_order_value: 775
Available total_order_value: 98666


In [23]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

In [24]:
orders[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_days"
    ]
].head(10)

,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,8.436574
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,13.782037
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,9.394213
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-02 00:28:42,13.208750
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-16 18:17:02,2.873877
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-09 21:57:05,2017-07-26 10:57:55,16.542245
6,136cce7faa42fdb2cefd53fdc79a6098,2017-04-11 12:22:08,NaT,NaN
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-16 13:10:30,2017-05-26 12:55:51,9.989826
8,76c6e866289321a7c93b82b54852dc33,2017-01-23 18:29:09,2017-02-02 14:08:10,9.818762
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-07-29 11:55:02,2017-08-16 17:14:30,18.221852


In [25]:
print("Missing delivery_days:", orders["delivery_days"].isna().sum())
print("Negative delivery_days:", (orders["delivery_days"] < 0).sum())

Missing delivery_days: 2965
Negative delivery_days: 0


In [26]:
orders["delivery_days"].describe()

count    96476.000000
mean        12.558702
std          9.546530
min          0.533414
25%          6.766403
50%         10.217755
75%         15.720327
max        209.628611
Name: delivery_days, dtype: float64

In [27]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

In [28]:
orders[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_delay_days"
    ]
].head(10)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,-7.107488
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,-5.355729
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,-17.245498
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,-12.980069
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,-9.238171
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-26 10:57:55,2017-08-01,-5.543113
6,136cce7faa42fdb2cefd53fdc79a6098,NaT,2017-05-09,NaN
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-26 12:55:51,2017-06-07,-11.461215
8,76c6e866289321a7c93b82b54852dc33,2017-02-02 14:08:10,2017-03-06,-31.410995
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-08-16 17:14:30,2017-08-23,-6.281597


In [29]:
orders["delivery_status"] = "Not Delivered"

orders.loc[
    orders["delivery_delay_days"] <= 0,
    "delivery_status"
] = "On Time"

orders.loc[
    orders["delivery_delay_days"] > 0,
    "delivery_status"
] = "Delayed"

In [30]:
orders["delivery_status"].value_counts()

delivery_status
On Time          88649
Delayed           7827
Not Delivered     2965
Name: count, dtype: int64

In [31]:
customer_order_count = (
    orders
    .groupby("customer_id")["order_id"]
    .count()
    .reset_index(name="customer_order_count")
)

In [32]:
customer_order_count.head(10)

,customer_id,customer_order_count
0,00012a2ce6f8dcda20d059ce98491703,1
1,000161a058600d5901f007fab4c27140,1
2,0001fd6190edaaf884bcaf3d49edf079,1
3,0002414f95344307404f0ace7a26f1d5,1
4,000379cdec625522490c315e70c7a9fb,1
5,0004164d20a9e969af783496f3408652,1
6,000419c5494106c306a97b5635748086,1
7,00046a560d407e99b969756e0b10f282,1
8,00050bf6e01e69d5c0fd612f1bcfb69c,1
9,000598caf2ef4117407665ac33275130,1


In [33]:
orders = orders.merge(
    customer_order_count,
    on="customer_id",
    how="left"
)

In [34]:
orders[["order_id", "customer_id", "customer_order_count"]].head(10)

,order_id,customer_id,customer_order_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,1
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,1
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,1
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,1
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,1
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,1
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,1


In [35]:
orders["customer_order_count"].isna().sum()

np.int64(0)

In [36]:
customer_spending = (
    orders
    .groupby("customer_id")["total_order_value"]
    .sum(min_count=1)
    .reset_index(name="customer_total_spending")
)

In [37]:
customer_spending.head(10)

,customer_id,customer_total_spending
0,00012a2ce6f8dcda20d059ce98491703,114.74
1,000161a058600d5901f007fab4c27140,67.41
2,0001fd6190edaaf884bcaf3d49edf079,195.42
3,0002414f95344307404f0ace7a26f1d5,179.35
4,000379cdec625522490c315e70c7a9fb,107.01
5,0004164d20a9e969af783496f3408652,71.80
6,000419c5494106c306a97b5635748086,49.40
7,00046a560d407e99b969756e0b10f282,166.59
8,00050bf6e01e69d5c0fd612f1bcfb69c,85.23
9,000598caf2ef4117407665ac33275130,1255.71


In [38]:
orders = orders.merge(
    customer_spending,
    on="customer_id",
    how="left"
)

In [39]:
orders[
    ["order_id", "customer_id", "customer_total_spending"]
].head(10)

,order_id,customer_id,customer_total_spending
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,28.62
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,175.26
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,65.95
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,75.16
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,35.95
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,169.76


In [40]:
orders["customer_total_spending"].isna().sum()

np.int64(775)

In [41]:
orders["average_order_value"] = (
    orders["customer_total_spending"]
    / orders["customer_order_count"]
)

In [42]:
orders[
    ["customer_id", "customer_order_count",
     "customer_total_spending", "average_order_value"]
].head(10)

,customer_id,customer_order_count,customer_total_spending,average_order_value
0,9ef432eb6251297304e76186b10a928d,1,38.71,38.71
1,b0830fb4747a6c6d20dea0b8c802d7ef,1,141.46,141.46
2,41ce2a54c0b03bf3443c3d931a367089,1,179.12,179.12
3,f88197465ea7920adcdbec7375364d82,1,72.20,72.20
4,8ab97904e6daea8866dbdbc4fb7aad2c,1,28.62,28.62
5,503740e9ca751ccdda7ba28e9ab8f608,1,175.26,175.26
6,ed0271e0b7da060a393796590e7b737a,1,65.95,65.95
7,9bdf08b4b3b52b5526ff42d37d47f222,1,75.16,75.16
8,f54a9f0e6b351c431402b8461ea51999,1,35.95,35.95
9,31ad1d1b63eb9962463f764d4e6e0c9d,1,169.76,169.76


In [43]:
orders["average_order_value"].isna().sum()

np.int64(775)

In [44]:
orders["repeat_customer"] = (
    orders["customer_order_count"] > 1
)

In [45]:
orders[
    ["customer_id", "customer_order_count", "repeat_customer"]
].head(10)

,customer_id,customer_order_count,repeat_customer
0,9ef432eb6251297304e76186b10a928d,1,False
1,b0830fb4747a6c6d20dea0b8c802d7ef,1,False
2,41ce2a54c0b03bf3443c3d931a367089,1,False
3,f88197465ea7920adcdbec7375364d82,1,False
4,8ab97904e6daea8866dbdbc4fb7aad2c,1,False
5,503740e9ca751ccdda7ba28e9ab8f608,1,False
6,ed0271e0b7da060a393796590e7b737a,1,False
7,9bdf08b4b3b52b5526ff42d37d47f222,1,False
8,f54a9f0e6b351c431402b8461ea51999,1,False
9,31ad1d1b63eb9962463f764d4e6e0c9d,1,False


In [46]:
orders["repeat_customer"].value_counts()

repeat_customer
False    99441
Name: count, dtype: int64

In [47]:
customers.columns.tolist()

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

In [48]:
orders = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

In [49]:
orders[
    ["customer_id", "customer_unique_id"]
].head(10)

,customer_id,customer_unique_id
0,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff
1,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231
2,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8
3,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977
4,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6
5,503740e9ca751ccdda7ba28e9ab8f608,80bb27c7c16e8f973207a5086ab329e2
6,ed0271e0b7da060a393796590e7b737a,36edbb3fb164b1f16485364b6fb04c73
7,9bdf08b4b3b52b5526ff42d37d47f222,932afa1e708222e5821dac9cd5db4cae
8,f54a9f0e6b351c431402b8461ea51999,39382392765b6dc74812866ee5ee92a7
9,31ad1d1b63eb9962463f764d4e6e0c9d,299905e3934e9e181bfb2e164dd4b4f8


In [50]:
orders["customer_unique_id"].isna().sum()

np.int64(0)

In [51]:
customer_unique_order_count = (
    orders
    .groupby("customer_unique_id")["order_id"]
    .count()
    .reset_index(name="customer_unique_order_count")
)

In [52]:
customer_unique_order_count.head(10)

,customer_unique_id,customer_unique_order_count
0,0000366f3b9a7992bf8c76cfdf3221e2,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1
2,0000f46a3911fa3c0805444483337064,1
3,0000f6ccb0745a6a4b88665a16c9f078,1
4,0004aac84e0df4da2b147fca70cf8255,1
5,0004bd2a26a76fe21f786e4fbd80607f,1
6,00050ab1314c0e55a6ca13cf7181fecf,1
7,00053a61a98854899e70ed204dd4bafe,1
8,0005e1862207bf6ccc02e4228effd9a0,1
9,0005ef4cd20d2893f0d9fbd94d3c0d97,1


In [53]:
(customer_unique_order_count["customer_unique_order_count"] > 1).sum()

np.int64(2997)

In [54]:
orders = orders.merge(
    customer_unique_order_count,
    on="customer_unique_id",
    how="left"
)

In [55]:
orders[
    ["customer_unique_id", "customer_unique_order_count"]
].head(10)

,customer_unique_id,customer_unique_order_count
0,7c396fd4830fd04220f754e42b4e5bff,2
1,af07308b275d755c9edb36a90c618231,1
2,3a653a41f6f9fc3d2a113cf8398680e8,1
3,7c142cf63193a1473d2e66489a9ae977,1
4,72632f0f9dd73dfee390c9b22eb56dd6,1
5,80bb27c7c16e8f973207a5086ab329e2,1
6,36edbb3fb164b1f16485364b6fb04c73,1
7,932afa1e708222e5821dac9cd5db4cae,1
8,39382392765b6dc74812866ee5ee92a7,1
9,299905e3934e9e181bfb2e164dd4b4f8,1


In [56]:
orders["customer_unique_order_count"].isna().sum()

np.int64(0)

In [57]:
orders["repeat_customer"] = (
    orders["customer_unique_order_count"] > 1
)

In [58]:
orders["repeat_customer"].value_counts()

repeat_customer
False    93099
True      6342
Name: count, dtype: int64

In [59]:
seller_order_count = (
    order_items
    .groupby("seller_id")["order_id"]
    .nunique()
    .reset_index(name="seller_order_count")
)

In [60]:
seller_order_count.head(10)

,seller_id,seller_order_count
0,0015a82c2db000af6aaaf3ae2ecb0532,3
1,001cca7ae9ae17fb1caed9dfb1094831,200
2,001e6ad469a905060d959994f1b41e4f,1
3,002100f778ceb8431b7a1020ff7ab48f,51
4,003554e2dce176b5555353e4f3555ac8,1
5,004c9cd9d87a3c30c522c48c4fc07416,158
6,00720abe85ba0859807595bbf045a33b,13
7,00ab3eff1b5192e5f1a63bcecfee11c8,1
8,00d8b143d12632bad99c0ad66ad52825,1
9,00ee68308b45bc5e2660cd833c3f81cc,135


In [61]:
seller_order_count["seller_order_count"].isna().sum()

np.int64(0)

In [62]:
seller_revenue = (
    order_items
    .groupby("seller_id")["price"]
    .sum()
    .reset_index(name="seller_revenue")
)

In [63]:
seller_revenue.head(10)

,seller_id,seller_revenue
0,0015a82c2db000af6aaaf3ae2ecb0532,2685.00
1,001cca7ae9ae17fb1caed9dfb1094831,25080.03
2,001e6ad469a905060d959994f1b41e4f,250.00
3,002100f778ceb8431b7a1020ff7ab48f,1234.50
4,003554e2dce176b5555353e4f3555ac8,120.00
5,004c9cd9d87a3c30c522c48c4fc07416,19712.71
6,00720abe85ba0859807595bbf045a33b,1007.50
7,00ab3eff1b5192e5f1a63bcecfee11c8,98.00
8,00d8b143d12632bad99c0ad66ad52825,86.00
9,00ee68308b45bc5e2660cd833c3f81cc,20260.00


In [64]:
seller_revenue["seller_revenue"].isna().sum()

np.int64(0)

In [65]:
sellers = sellers.merge(
    seller_order_count,
    on="seller_id",
    how="left"
)

sellers = sellers.merge(
    seller_revenue,
    on="seller_id",
    how="left"
)

In [66]:
sellers[
    ["seller_id", "seller_order_count", "seller_revenue"]
].head(10)

,seller_id,seller_order_count,seller_revenue
0,3442f8959a84dea7ee197c632cb2df15,3,218.70
1,d1b65fc7debc3361ea86b5f14c68d2e2,40,11703.07
2,ce3ad9de960102d0677a81f5d0bb7b2d,1,158.00
3,c0f3eea2e14555b6faeea3dd58c1b1c3,1,79.99
4,51a04a8a6bdcb23deccc82b0b80742cf,1,167.99
5,c240c4061717ac1806ae6ee72be3533b,1,59.90
6,e49c26c3edfa46d227d5121a6b6e4d37,35,3654.25
7,1b938a7ec6ac5061a66a3766e0e75f90,30,3987.60
8,768a86e36ad6aae3d03ee3c6433d61df,17,587.46
9,ccc4bbb5f32a6ab2b7066a4130f114e3,187,74004.62


In [67]:
sellers[
    ["seller_order_count", "seller_revenue"]
].isna().sum()

seller_order_count    0
seller_revenue        0
dtype: int64

In [68]:
orders.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_days',
 'delivery_delay_days',
 'total_order_value',
 'delivery_status',
 'customer_order_count',
 'customer_total_spending',
 'average_order_value',
 'repeat_customer',
 'customer_unique_id',
 'customer_unique_order_count']

In [69]:
sellers.columns.tolist()

['seller_id',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state',
 'seller_order_count',
 'seller_revenue']

In [70]:
print("Orders:", len(orders))
print("Sellers:", len(sellers))

Orders: 99441
Sellers: 3095


In [71]:
print("Duplicate order_id:", orders["order_id"].duplicated().sum())
print("Duplicate seller_id:", sellers["seller_id"].duplicated().sum())

Duplicate order_id: 0
Duplicate seller_id: 0


In [72]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [73]:
orders.to_csv(
    "../data/processed/orders_features.csv",
    index=False
)

In [74]:
sellers.to_csv(
    "../data/processed/sellers_features.csv",
    index=False
)

In [75]:
order_items.to_csv(
    "../data/processed/order_items_features.csv",
    index=False
)